# **マルコフじゃんけん**

## **概要**
対戦を通してプレイヤーの癖を学習し、自動で強くなっていくジャンケンCPUです。


プレイヤーが直前に出した手を記憶し次に出す手を確率で決定します。学習の仕組みは各手の初期値を「1」とし、AIが勝つたびにその手のカウントを「+1」と加算していきます。このカウントをそのまま手を出す重み（確率）として利用するため、勝率の高い手が自然と選ばれやすくなります。


複雑な計算やライブラリを使わず、リストの足し算のみで確率を調整する理解しやすい設計が特徴です。対戦を重ねるほどプレイヤーの行動傾向を分析し、最善手を選択する確率が高まっていきます。

なるべくシンプルなルールで学習を再現できるためrandomモジュールだけで実現することができました。

本来はclassを使うほうが良さそうですが理解できなかったのでマッピングしてそこから参照しています。
## **操作方法**

ユーザ入力の方法：

セルのすぐ下に入力用のテキストボックスが表示されます。
ボックス内に半角英数で gやc,p などを入力し、Enter キー を押すと対戦が1回進みます。
終了する際は q を入力して Enter キーを押すか、セルの停止ボタンを押します。

## **制作者のコメント**

このコードのポイント

1. 標準ライブラリのみで動作

   高度な機械学習ライブラリや複雑な数学関数（指数関数や対数など）を一切使わずPython標準のrandomモジュールのみで動作する点です。特別な環境構築が不要で初心者でもコードの全容を簡単に理解できます。


2. 「2次元リスト」による直感的なデータ管理

   データ管理にクラス定義classを使用せず、直感的な「2次元リスト」のみで完結している点です。「相手の1手前」を行、「CPUの手」を列とする一覧表のような構造になっているため、プログラム内でデータがどのように保持・参照されているかが視覚的にも明快です。


3. random.choicesによる確率の自動制御

   手の選択にrandom.choicesの重み付けweights機能を活用している点です。これにより、自分で複雑な確率計算の処理を書くことなく、勝った回数の多い手ほど自動的に選ばれやすくなります。

最小限のコード量と非常に素朴なロジックだけで強化学習の本質を実現している点が、本プログラム最大の魅力です。


## **作成時間**

作成時間：**約15時間**

In [ ]:
import random

HANDS = ["グー", "チョキ", "パー"]
INPUT_MAP = {"g": 0, "c": 1, "p": 2, "0": 0, "1": 1, "2": 2}


win_counts = [
    [1, 1, 1],  # 直前にグー
    [1, 1, 1],  # 直前にチョキ
    [1, 1, 1]   # 直前にパー
]

last_player_hand = None
game_count = 0
ai_wins = 0

print("============================================")
print(" 【マルコフジャンケン】")
print(" 入力: g(グー) / c(チョキ) / p(パー) または 0 / 1 / 2")
print("============================================")

while True:
    print(f"\n--- 第 {game_count + 1} 回戦 ---")

    user_input = input("あなたの手を入力してENTER: ").strip().lower()

    if user_input == 'q':
        print("\n対戦を終了します。")
        break
    if user_input not in INPUT_MAP:
        print("正しい手を入力してください (g, c, p, 0, 1, 2)")
        continue

    player_hand = INPUT_MAP[user_input]

    # 2. AIのだすて
    if last_player_hand is None:
        ai_hand = random.choice([0, 1, 2])
    else:
        counts = win_counts[last_player_hand]
        ai_hand = random.choices([0, 1, 2], weights=counts)[0]

    #  勝敗判定
    if ai_hand == player_hand:
        result_str = "あいこ！"
        is_ai_win = False
    elif (ai_hand == 0 and player_hand == 1) or \
         (ai_hand == 1 and player_hand == 2) or \
         (ai_hand == 2 and player_hand == 0):
        result_str = "AIの勝ち！"
        ai_wins += 1
        is_ai_win = True
    else:
        result_str = "あなたの勝ち！"
        is_ai_win = False

    game_count += 1
    print(f" あなた: {HANDS[player_hand]}  vs  AI: {HANDS[ai_hand]}")
    print(f" 結果: {result_str}")


    if last_player_hand is not None and is_ai_win:
        win_counts[last_player_hand][ai_hand] += 1

    last_player_hand = player_hand

    current_counts = win_counts[player_hand]
    total = sum(current_counts)
    p0 = (current_counts[0] / total) * 100
    p1 = (current_counts[1] / total) * 100
    p2 = (current_counts[2] / total) * 100

    print(f"\n [AIの次の手札確率] あなたが【{HANDS[player_hand]}】を出した次:")
    print(f"   グー: {p0:.1f}% | チョキ: {p1:.1f}% | パー: {p2:.1f}%")

print(f"\n【最終結果】 全 {game_count} 戦中 AIの勝利数: {ai_wins} 勝")

 【マルコフジャンケン】
 入力: g(グー) / c(チョキ) / p(パー) または 0 / 1 / 2

--- 第 1 回戦 ---
 あなた: グー  vs  AI: パー
 結果: AIの勝ち！

 [AIの次の手札確率] あなたが【グー】を出した次:
   グー: 33.3% | チョキ: 33.3% | パー: 33.3%

--- 第 2 回戦 ---
 あなた: チョキ  vs  AI: グー
 結果: AIの勝ち！

 [AIの次の手札確率] あなたが【チョキ】を出した次:
   グー: 33.3% | チョキ: 33.3% | パー: 33.3%

--- 第 3 回戦 ---
 あなた: チョキ  vs  AI: チョキ
 結果: あいこ！

 [AIの次の手札確率] あなたが【チョキ】を出した次:
   グー: 33.3% | チョキ: 33.3% | パー: 33.3%

--- 第 4 回戦 ---
 あなた: チョキ  vs  AI: チョキ
 結果: あいこ！

 [AIの次の手札確率] あなたが【チョキ】を出した次:
   グー: 33.3% | チョキ: 33.3% | パー: 33.3%

--- 第 5 回戦 ---
 あなた: パー  vs  AI: グー
 結果: あなたの勝ち！

 [AIの次の手札確率] あなたが【パー】を出した次:
   グー: 33.3% | チョキ: 33.3% | パー: 33.3%

--- 第 6 回戦 ---
 あなた: グー  vs  AI: グー
 結果: あいこ！

 [AIの次の手札確率] あなたが【グー】を出した次:
   グー: 50.0% | チョキ: 25.0% | パー: 25.0%

--- 第 7 回戦 ---
 あなた: チョキ  vs  AI: チョキ
 結果: あいこ！

 [AIの次の手札確率] あなたが【チョキ】を出した次:
   グー: 33.3% | チョキ: 33.3% | パー: 33.3%

--- 第 8 回戦 ---
 あなた: グー  vs  AI: パー
 結果: AIの勝ち！

 [AIの次の手札確率] あなたが【グー】を出した次:
   グー: 50.0% | チョキ: 25.0% | パー: 25.0%

--- 第 9 回戦 -

# **ライフゲーム(没案)**
# **理由AIの使いすぎ**
## **概要**
ライフゲームとは、数学者コンウェイが考案した生命シミュレーションゲームです。

格子状のセルの「生・死」が、周囲の状況に応じて変化します。第0世代の配置を決めた後は見守るだけです。

かの有名なノイマンの理論をベースにしたシンプルなルールから

まるで生きているかのような複雑な動きが生まれるのが特徴です。

ルール(アルゴリズム)

これらのルールに則り世代が進んでいきます。
白色を死、黒色を生としています。


誕生
死んでいるセルに隣接する生きたセルがちょうど3つあれば、次の世代が誕生する。

生存
生きているセルに隣接する生きたセルが2つか3つならば、次の世代でも生存する。

過疎
生きているセルに隣接する生きたセルが1つ以下ならば、過疎により死滅する。

過密
生きているセルに隣接する生きたセルが4つ以上ならば、過密により死滅する。

wikipediaより


## **操作方法**
白黒のセルで第0世代のセルの配置を確定します。
確定したらスタートボタンを押してください。
スタートボタンを押すと1秒ごとに1世代づつ進んでいきます。


## **制作者のコメント**

二次元ベクトルの勉強になると思い作成しました。ライフゲームの世界をみて感動したのが理由でもあります。

上記にもある通り少ないシンプルなルールで構成されていますが特定の配置から世代交代をすすめると芸術的な挙動をするのが私の好きな部分です。

私が一番好きな振動子は銀河です。

超シンプルな生のセルの棒が4つ並べられたような形から形がくずれたような風車のような動きをしていて印象的でした
世代交代している途中に止めるとまた同じ形に戻っていてそれがループすることで振動子として動いているそうです。

ライフゲームはコンピュータが存在する前から存在している理論（？）でアメリカの大学で研究が盛んに行われたそうです


(コンピュータ上でなくても紙上でシミュレーションできる)。

ライフゲームはとても面白い学問なので多くの人に広まってほしいです。

ライブラリの関数の仕様を調べるのがきつかったです。

## **作成時間**

作成時間：**約8時間**

In [ ]:
import time
import numpy as np
from ipywidgets import Button, HBox, VBox, Output, Layout
from IPython.display import display, clear_output

class ColabLifeGame:
    def __init__(self, rows=20, cols=20):
        self.rows = rows
        self.cols = cols
        self.is_running = False

        # 0: 死 (白 / #ffffff), 1: 生 (黒 / #000000)
        self.grid = np.zeros((rows, cols), dtype=int)

        # セル用のボタンを配置する二次元配列
        self.cell_buttons = []

        # 1マスごとのボタンを作成 (サイズは20x20ピクセル)
        cell_layout = Layout(width='20px', height='20px', margin='1px')

        for r in range(rows):
            row_buttons = []
            for c in range(cols):
                btn = Button(layout=cell_layout)
                # 初期状態は白
                btn.style.button_color = '#ffffff'
                # クリック時に座標(r, c)を渡すための設定
                btn.click_callback = (r, c)
                btn.on_click(self.on_cell_click)
                row_buttons.append(btn)
            self.cell_buttons.append(row_buttons)

        # 操作用UIボタン
        self.start_btn = Button(description="スタート", button_style="success")
        self.start_btn.on_click(self.start_simulation)


        # 画面の組み立て
        controls = HBox([self.start_btn])
        grid_rows = [HBox(row) for row in self.cell_buttons]

        # コントロールボタンとマス目を縦に並べて表示
        display(VBox([controls, VBox(grid_rows)]))

    def update_display(self):
        """現在の内部データをボタンの色に反映する"""
        for r in range(self.rows):
            for c in range(self.cols):
                if self.grid[r, c] == 1:
                    self.cell_buttons[r][c].style.button_color = '#' # 生
                else:
                    self.cell_buttons[r][c].style.button_color = '#ffffff' # 死

    def on_cell_click(self, b):
        if self.is_running:
            return
        r, c = b.click_callback
        self.grid[r, c] = 1 - self.grid[r, c]
        self.update_display()

    def update_generation(self):
        """ライフゲームのルール（アルゴリズム）に基づいて次世代を計算"""
        next_grid = self.grid.copy()
        for r in range(self.rows):
            for c in range(self.cols):
                # 周囲8マスの生存数をカウント
                neighbors = self.grid[max(0, r-1):min(self.rows, r+2), max(0, c-1):min(self.cols, c+2)]
                live_neighbors = np.sum(neighbors) - self.grid[r, c]

                if self.grid[r, c] == 1:
                    # 【過疎・過密】 生存数が1以下、または4以上なら死滅
                    if live_neighbors <= 1 or live_neighbors >= 4:
                        next_grid[r, c] = 0
                else:
                    # 【誕生】 死んでいるセルに隣接する生きたセルがちょうど3つ
                    if live_neighbors == 3:
                        next_grid[r, c] = 1
        self.grid = next_grid

    def start_simulation(self, b):
        """スタートボタンでループ処理を開始"""
        if self.is_running:
            return
        self.is_running = True

        # スタートしたらセルのボタンをクリックできないようにする
        while self.is_running:
            self.update_generation()
            self.update_display()
            time.sleep(1.0) # 1秒待機


# アプリケーションの実行
app = ColabLifeGame(20, 20)